# Voxis: English and Hindi speech recognition with wav2vec2

[![Open In Colab](https://colab.research.google.com/assets/colab/badge.svg)](https://colab.research.google.com/github/<your-github-username>/Voxis/blob/main/notebooks/ASR_Finetuning_Final.ipynb)

This notebook contains the full experimental pipeline behind the Voxis app: evaluation harness, English study, Hindi fine-tuning, model export, and the comparison against Whisper and NVIDIA NeMo.

## Final models (frozen, served by the backend)
| Language | Model | How it was obtained |
|---|---|---|
| English | `facebook/wav2vec2-large-960h-lv60-self`, greedy CTC | pretrained checkpoint, selected by evaluation (Part C) |
| Hindi | `Harveenchadha/vakyansh-wav2vec2-hindi-him-4200` fine-tuned on Google FLEURS `hi_in` | our fine-tuning (Part D) |

## Results (all numbers from the harness in `asr_common.py`: same files, same text normaliser for reference and prediction)
| Language | Model | Test set | WER % | CER % |
|---|---|---|---|---|
| English | wav2vec2-base-960h (pretrained) | LibriSpeech test-clean (2,620) | 3.26 | 0.96 |
| English | wav2vec2-base-960h (pretrained) | LibriSpeech test-other (2,939) | 8.43 | 3.11 |
| English | **wav2vec2-large-960h-lv60-self** | LibriSpeech test-clean | **1.75** | 0.50 |
| English | **wav2vec2-large-960h-lv60-self** | LibriSpeech test-other | **3.73** | 1.27 |
| English | wav2vec2-base-960h / large-lv60-self | AppTek call-centre, 20 files (cross-domain) | 30.05 / **18.61** | 13.94 / 8.14 |
| Hindi | **Vakyansh, fine-tuned by us** | FLEURS hi_in test (418) | **19.33** | **7.20** |
| Hindi | Whisper-large-v3 | FLEURS hi_in test (418) | 28.75 | 10.82 |
| Hindi | our first pilot (under 1 epoch, superseded) | FLEURS hi_in, 300 clips | 30.34 | 13.14 |

Whisper-large-v3 and NeMo on English under this harness are in Part F.

## Notebook map
| Part | Content | Needs GPU |
|---|---|---|
| A | Setup: packages, Hugging Face login, shared helper module, Hub backup | no |
| B | AppTek dataset resolver | no |
| C | English: pretrained base vs large backbone (optional LM decoding) | yes |
| D | Hindi: Whisper baseline, fine-tuning, evaluation | yes (about 1 h) |
| E | Freeze and export both models to a private Hub repo | yes |
| F | Remaining comparisons: Whisper (English) and NeMo (run NeMo in a fresh runtime) | yes |
| G | How to read the results and limitations | no |
| Appendix | Earlier experiments (English LibriSpeech fine-tune, conversational fine-tune, first Hindi pilot, old Whisper/NeMo cells). Kept for the report; not needed to reproduce the final results | yes |

**Run order:** Part A, then B, C, D, E in order. Colab note: results are written to `/content/results` and every result file is pushed to a private Hugging Face repo, because a runtime disconnect wipes `/content`.

## Part A. Setup
Installs the packages, logs in to Hugging Face (Colab secret `HUGGING_FACE_TOKEN`, a **write** token), writes the shared helper module `asr_common.py` (the same file the FastAPI backend imports, so training-time and serving-time text and audio handling are identical), and creates the private backup repo.

In [ ]:
# Core packages for Wav2Vec2 + Whisper evaluation/training
!pip install -q -U transformers datasets evaluate jiwer soundfile librosa accelerate huggingface_hub

!pip install --upgrade "numpy>=2.0.0"

In [ ]:
# Hugging Face authentication is optional for public checkpoints/datasets.
# If you stored a token in Colab Secrets as HUGGING_FACE_TOKEN, this logs in.
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get("HUGGING_FACE_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face.")
    else:
        print("No HUGGING_FACE_TOKEN secret found; continuing without login.")
except Exception as exc:
    print(f"Skipping Hugging Face login: {exc}")

### A.3 Shared helper module (`asr_common.py`)
Text normalisation (English: Whisper-style; Hindi: Devanagari-aware), 16 kHz audio loading, chunked CTC inference, WER/CER scoring with substitution/deletion/insertion breakdown, result saving. Source of truth: `backend/asr_common.py` in the repo.

In [ ]:
%%writefile asr_common.py
"""asr_common.py - shared ASR helpers.

Used now for trustworthy evaluation in the notebook, and later imported by the
FastAPI server so that training-time and serving-time text/audio handling are
identical.

Contents
  1. Text normalisation (English: Whisper-style, Hindi: Devanagari-aware)
  2. Audio loading -> mono / 16 kHz / float32
  3. Long-audio-safe CTC inference (chunking + stride, no OOM on 7-minute calls)
  4. Scoring (WER / CER + substitution / deletion / insertion breakdown)
  5. Saving / re-scoring results as JSON so nothing is lost on a Colab disconnect
"""
from __future__ import annotations

import json
import re
import time
import unicodedata
from pathlib import Path

import numpy as np

TARGET_SR = 16_000


# ----------------------------------------------------------------------------
# 1. Text normalisation
# ----------------------------------------------------------------------------
_FILLERS = re.compile(r"\b(hmm|mm|mhm|mmm|uh|um)\b")
_CUT_WORD = re.compile(r"\b\w+~")          # AppTek marks cut-off words as "PRE~"

EN_NORMALIZER_NAME = None
_EN_NORM = None


def _simple_en(text: str) -> str:
    """Fallback if the Whisper normaliser cannot be loaded."""
    text = str(text or "").lower()
    text = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", text)
    text = _FILLERS.sub(" ", text)
    text = text.replace("'", "")
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return " ".join(text.split())


def _get_en_normalizer():
    global _EN_NORM, EN_NORMALIZER_NAME
    if _EN_NORM is not None:
        return _EN_NORM
    try:
        from transformers import WhisperTokenizer
        from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

        tok = WhisperTokenizer.from_pretrained("openai/whisper-small")
        _EN_NORM = EnglishTextNormalizer(tok.english_spelling_normalizer)
        EN_NORMALIZER_NAME = "whisper_english_text_normalizer"
    except Exception as exc:  # noqa: BLE001
        print(f"[asr_common] Whisper normaliser unavailable ({exc!r}); using simple fallback.")
        _EN_NORM = _simple_en
        EN_NORMALIZER_NAME = "simple_fallback"
    print(f"[asr_common] English normaliser: {EN_NORMALIZER_NAME}")
    return _EN_NORM


def normalize_en(text: str) -> str:
    """Lower-case, strip punctuation/fillers/bracketed tags, spelled numbers -> digits,
    British -> American spelling. Apply to BOTH reference and hypothesis."""
    text = unicodedata.normalize("NFKC", str(text or ""))
    text = _CUT_WORD.sub(" ", text)
    return " ".join(_get_en_normalizer()(text).split())


def normalize_hi(text: str) -> str:
    """Devanagari-aware normalisation (same logic as the Hindi training cell)."""
    text = unicodedata.normalize("NFC", str(text or ""))
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat == "Cf":                                  # invisible formatting chars
            continue
        if "LATIN" in unicodedata.name(ch, "") and ord(ch) > 127:
            ascii_letters = "".join(
                c for c in unicodedata.normalize("NFKD", ch) if c.isascii() and c.isalpha()
            )
            out.append(ascii_letters.lower() if ascii_letters else " ")
            continue
        if ch.isascii() and ch.isalpha():
            out.append(ch.lower())
            continue
        if cat.startswith("P") or cat.startswith("S"):   # punctuation/symbols -> separator
            out.append(" ")
            continue
        out.append(ch)
    return " ".join("".join(out).split())


NORMALIZERS = {"en": normalize_en, "hi": normalize_hi}


# ----------------------------------------------------------------------------
# 2. Audio
# ----------------------------------------------------------------------------
def to_mono_16k(audio, sr: int) -> np.ndarray:
    """Any (samples,) / (samples, ch) / (ch, samples) array -> mono float32 @ 16 kHz."""
    import librosa

    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=1) if audio.shape[0] > audio.shape[1] else audio.mean(axis=0)
    elif audio.ndim > 2:
        raise ValueError(f"Unsupported audio shape {audio.shape}")
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
    return np.ascontiguousarray(audio, dtype=np.float32)


def load_audio_file(path: str) -> np.ndarray:
    """Read a wav/flac/ogg/... file -> mono 16 kHz float32."""
    import soundfile as sf

    audio, sr = sf.read(path, dtype="float32")
    return to_mono_16k(audio, sr)


# ----------------------------------------------------------------------------
# 3. Long-audio-safe CTC inference
# ----------------------------------------------------------------------------
def load_ctc_pipeline(model_dir: str, device=None, chunk_s: float = 20.0,
                      stride_s=(4, 2), batch_size: int = 8,
                      lm_path: str | None = None, alpha: float = 0.5, beta: float = 1.5):
    """Wav2Vec2-CTC ASR pipeline that splits long audio into overlapping chunks.

    Wav2Vec2 attention memory grows quadratically with length, so feeding a
    7-minute call in one forward pass (what the old AppTek cell did) OOMs a T4
    and degrades quality. Chunking with stride fixes both.
    """
    import torch
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, pipeline

    if device is None:
        device = 0 if torch.cuda.is_available() else -1
    processor = Wav2Vec2Processor.from_pretrained(model_dir)
    model = Wav2Vec2ForCTC.from_pretrained(model_dir).eval()
    # "group"-norm models (wav2vec2-base family) have NO attention mask, so zero-padding
    # inside a batch corrupts shorter inputs (huge deletion rates). Batch only
    # "layer"-norm models (large / lv60 / XLSR), which support attention masks.
    if getattr(model.config, "feat_extract_norm", "layer") == "group":
        batch_size = 1
    extra = {}
    if lm_path:   # n-gram (KenLM) beam-search decoding via pyctcdecode
        from pyctcdecode import build_ctcdecoder

        vocab = processor.tokenizer.get_vocab()
        labels = [t for t, _ in sorted(vocab.items(), key=lambda kv: kv[1])]
        extra["decoder"] = build_ctcdecoder(labels=labels, kenlm_model_path=lm_path,
                                            alpha=alpha, beta=beta)
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
        chunk_length_s=chunk_s,
        stride_length_s=stride_s,
        batch_size=batch_size,
        **extra,
    )
    pipe.asr_batch_size = batch_size       # remembered for transcribe_many
    return pipe


def transcribe_ctc(pipe, audio_16k: np.ndarray) -> str:
    return pipe({"raw": audio_16k, "sampling_rate": TARGET_SR})["text"]


def transcribe_many(pipe, arrays, batch_size=None):
    batch_size = batch_size or getattr(pipe, "asr_batch_size", 1)
    inputs = [{"raw": a, "sampling_rate": TARGET_SR} for a in arrays]
    return [o["text"] for o in pipe(inputs, batch_size=batch_size)]


# ----------------------------------------------------------------------------
# 4. Scoring
# ----------------------------------------------------------------------------
def score(preds, refs, normalizer=normalize_en) -> dict:
    """Corpus-level WER/CER (percent) plus S/D/I breakdown (percent of reference words).

    High deletions  -> model dropped speech (truncation / chunking problem).
    High insertions -> hallucination / repetition.
    High substitutions -> genuine recognition errors.
    """
    import jiwer

    P, R, kept = [], [], []
    for i, (p, r) in enumerate(zip(preds, refs)):
        r_n = normalizer(r)
        if not r_n:
            continue
        P.append(normalizer(p))
        R.append(r_n)
        kept.append(i)
    if not R:
        return {"n": 0}
    w = jiwer.process_words(R, P)
    n_ref_words = w.hits + w.substitutions + w.deletions
    return {
        "n": len(R),
        "wer": round(w.wer * 100, 2),
        "cer": round(jiwer.cer(R, P) * 100, 2),
        "sub_pct": round(100 * w.substitutions / n_ref_words, 2),
        "del_pct": round(100 * w.deletions / n_ref_words, 2),
        "ins_pct": round(100 * w.insertions / n_ref_words, 2),
        "per_sample_wer": [round(jiwer.wer(r, p) * 100, 1) for r, p in zip(R, P)],
        "kept_indices": kept,
    }


def print_score(title: str, s: dict) -> None:
    if not s.get("n"):
        print(f"{title}: no valid samples")
        return
    print(f"{title}\n  samples={s['n']}  WER={s['wer']}%  CER={s['cer']}%  "
          f"(sub {s['sub_pct']}% / del {s['del_pct']}% / ins {s['ins_pct']}%)")


# ----------------------------------------------------------------------------
# 5. Persist / re-score results
# ----------------------------------------------------------------------------
def save_results(path, *, model: str, dataset: str, ids, preds, refs,
                 audio_seconds: float | None = None, elapsed_seconds: float | None = None,
                 lang: str = "en", extra: dict | None = None) -> None:
    """Store RAW predictions and references (un-normalised) so any run can be
    re-scored later with an improved normaliser without re-running the model."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model": model,
        "dataset": dataset,
        "lang": lang,
        "n": len(ids),
        "ids": list(ids),
        "preds_raw": list(preds),
        "refs_raw": list(refs),
        "audio_seconds": audio_seconds,
        "elapsed_seconds": elapsed_seconds,
        "rtf": (elapsed_seconds / audio_seconds) if audio_seconds and elapsed_seconds else None,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "extra": extra or {},
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[asr_common] saved {len(ids)} results -> {path}")


def rescore(path) -> dict:
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    s = score(d["preds_raw"], d["refs_raw"], NORMALIZERS[d.get("lang", "en")])
    s.update(model=d["model"], dataset=d["dataset"], rtf=d.get("rtf"))
    return s


def compare(paths) -> None:
    """Print a one-line-per-run table from saved JSON results."""
    print(f"{'model':45s} {'dataset':32s} {'n':>5s} {'WER%':>7s} {'CER%':>7s} {'del%':>6s} {'ins%':>6s}")
    for p in paths:
        s = rescore(p)
        print(f"{s['model'][:45]:45s} {s['dataset'][:32]:32s} {s['n']:5d} "
              f"{s['wer']:7.2f} {s['cer']:7.2f} {s['del_pct']:6.2f} {s['ins_pct']:6.2f}")

### A.4 Load the helpers and set up Hub backup

In [ ]:
# 1.0  Shared helpers + backup to the Hugging Face Hub (no Google Drive needed).
# Upload asr_common.py to /content first (Files panel on the left -> upload).
import sys, importlib, time, gc, os, glob, torch
import numpy as np
sys.path.insert(0, "/content")
import asr_common
importlib.reload(asr_common)
from asr_common import *

RESULTS_DIR = "/content/results"            # small JSON files, a few KB each
os.makedirs(RESULTS_DIR, exist_ok=True)

from huggingface_hub import HfApi
_api = HfApi()
BACKUP_REPO = _api.create_repo("asr-project-backup", repo_type="model", private=True,
                               exist_ok=True).repo_id        # e.g. "your-username/asr-project-backup"

def backup_file(local_path, path_in_repo=None):
    """Push one small file (e.g. a results JSON) to the private Hub repo."""
    _api.upload_file(path_or_fileobj=local_path,
                     path_in_repo=path_in_repo or os.path.basename(local_path),
                     repo_id=BACKUP_REPO, repo_type="model")
    print("backed up ->", BACKUP_REPO, "/", path_in_repo or os.path.basename(local_path))

def backup_folder(local_dir, path_in_repo):
    """Push a whole model folder (skips optimizer/checkpoint junk)."""
    _api.upload_folder(folder_path=local_dir, path_in_repo=path_in_repo, repo_id=BACKUP_REPO,
                       repo_type="model", ignore_patterns=["checkpoint-*", "*.bin", "optimizer.pt", "training_args.bin"])
    print("backed up folder ->", BACKUP_REPO, "/", path_in_repo)

print("English normaliser:", asr_common._get_en_normalizer.__name__)
print("Backup repo:", BACKUP_REPO)

## Part B. AppTek call-centre dataset
Used only as a cross-domain stress test for English (phone calls, 7-15 minute recordings). Audio is resolved by file name; ambiguous paths are skipped, never guessed.

In [ ]:
# ==========================================
# AppTek dataset inspection + shared audio-file resolver
# ==========================================
import os
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

APPTEK_REPO_ID = "apptek-com/apptek_callcenter_dialogues"
APPTEK_CONFIG = "default"
APPTEK_SPLIT = "test"

apptek_metadata = load_dataset(
    APPTEK_REPO_ID, APPTEK_CONFIG, split=APPTEK_SPLIT
)
print("Dataset columns:", apptek_metadata.column_names)
print("First sample:", apptek_metadata[0])

# Build a filename -> all matching paths map once. This avoids silently
# selecting an arbitrary file when a basename occurs in multiple folders.
_apptek_api = HfApi()
_apptek_repo_files = _apptek_api.list_repo_files(
    repo_id=APPTEK_REPO_ID, repo_type="dataset"
)
APPTEK_AUDIO_PATHS = {}
for _path in _apptek_repo_files:
    if _path.lower().endswith((".wav", ".flac", ".mp3")) and _path.startswith("test/"):
        APPTEK_AUDIO_PATHS.setdefault(os.path.basename(_path), []).append(_path)

def resolve_apptek_audio_path(file_name, accent=None):
    """Resolve one unambiguous test audio path; return None if ambiguous/missing."""
    base = os.path.basename(str(file_name or ""))
    matches = APPTEK_AUDIO_PATHS.get(base, [])
    if len(matches) == 1:
        return matches[0]
    # Prefer the en-AU folder when it is the unique matching channel/path.
    en_au = [x for x in matches if "/en-AU/" in x]
    if len(en_au) == 1:
        return en_au[0]
    if accent:
        accent_norm = str(accent).lower().replace("_", "-")
        accent_matches = [
            x for x in matches
            if accent_norm in x.lower().replace("_", "-")
        ]
        if len(accent_matches) == 1:
            return accent_matches[0]
    return None

def download_apptek_audio(file_name, accent=None, cache_dir="/content/apptek_audio"):
    path = resolve_apptek_audio_path(file_name, accent=accent)
    if path is None:
        return None
    os.makedirs(cache_dir, exist_ok=True)
    return hf_hub_download(
        repo_id=APPTEK_REPO_ID,
        filename=path,
        repo_type="dataset",
        cache_dir=cache_dir
    )

print("AppTek audio resolver ready. Ambiguous/missing paths are skipped, not guessed.")

## Part C. English
**C.1 Evaluation harness check.** Fine-tuned/pretrained wav2vec2 is evaluated with 20 s chunks (4 s left / 2 s right stride), a Whisper-style normaliser applied to reference and prediction, and the full LibriSpeech test sets. `wav2vec2-base` has no attention mask, so it is run with batch size 1 (padding inside a batch corrupts it; this was a real bug we found and fixed). Reproducing the published base-model numbers (about 3.4 / 8.6) validates the harness.

In [ ]:
APPTEK_ROWS = range(20)                      # rows 0-19, identical to Whisper / NeMo runs
# The old fine-tune is gone and was not better than the pretrained model, so the
# pretrained checkpoint is the baseline for now.
EN_FT_DIR   = "facebook/wav2vec2-base-960h"

w2v_pipe = load_ctc_pipeline(EN_FT_DIR, chunk_s=20, stride_s=(4, 2), batch_size=8)

ids, refs, preds = [], [], []
audio_secs, t0 = 0.0, time.time()
for r in APPTEK_ROWS:
    row = apptek_metadata[r]
    path = download_apptek_audio(row["file_name"], accent=row.get("accent"), cache_dir="/content/apptek_audio")
    if path is None:
        print("skip (unresolved path):", row["file_name"]); continue
    audio = load_audio_file(path)
    audio_secs += len(audio) / TARGET_SR
    preds.append(transcribe_ctc(w2v_pipe, audio))
    refs.append(row["text"])
    ids.append(row["file_name"])
    print(f"row {r:2d} | {len(audio)/TARGET_SR:6.1f}s | pred: {preds[-1][:70]}")
elapsed = time.time() - t0

save_results(f"{RESULTS_DIR}/apptek_w2v_en_base960h.json",
             model=EN_FT_DIR, dataset="apptek test rows 0-19", ids=ids, preds=preds, refs=refs,
             audio_seconds=audio_secs, elapsed_seconds=elapsed, lang="en")
print_score("Wav2Vec2-base-960h (English) on AppTek, chunked, Whisper-style normalisation",
            score(preds, refs, normalize_en))

backup_file(f"{RESULTS_DIR}/apptek_w2v_en_base960h.json")
del w2v_pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from datasets import load_dataset, Audio

MODELS = {
    "pretrained_base_960h": "facebook/wav2vec2-base-960h",
    # Step 2 will add the larger backbone here, e.g.:
    # "large_960h_lv60_self": "facebook/wav2vec2-large-960h-lv60-self",
}
SUBSETS = {"clean": 2620, "other": 2939}     # config -> number of test utterances

for subset, n_utts in SUBSETS.items():
    ds = (load_dataset("openslr/librispeech_asr", subset, split="test", streaming=True)
          .cast_column("audio", Audio(sampling_rate=16000)))
    samples = list(ds.take(n_utts))
    arrays  = [np.asarray(s["audio"]["array"], dtype=np.float32) for s in samples]
    refs_ls = [s["text"] for s in samples]
    ids_ls  = [s["id"] for s in samples]
    secs    = sum(len(a) for a in arrays) / TARGET_SR

    for name, model_dir in MODELS.items():
        pipe = load_ctc_pipeline(model_dir, batch_size=16)   # auto-forced to 1 for base models
        t0 = time.time()
        hyp = transcribe_many(pipe, arrays)
        el = time.time() - t0
        save_results(f"{RESULTS_DIR}/librispeech_test_{subset}__{name}.json",
                     model=model_dir, dataset=f"librispeech test-{subset} (full)", ids=ids_ls,
                     preds=hyp, refs=refs_ls, audio_seconds=secs, elapsed_seconds=el, lang="en")
        print_score(f"{name} | test-{subset}", score(hyp, refs_ls, normalize_en))
        backup_file(f"{RESULTS_DIR}/librispeech_test_{subset}__{name}.json")
        del pipe; gc.collect(); torch.cuda.empty_cache()
    del samples, arrays

**C.2 Larger backbone.** Fine-tuning the base model on LibriSpeech gave no real gain (it was already trained on all 960 h; see Appendix A.2). The lever that works is a stronger pretrained backbone: `wav2vec2-large-960h-lv60-self` (pretrained on 60k hours, with self-training).

In [ ]:
from datasets import load_dataset, Audio
_LS_CACHE = {}

def get_librispeech(subset, n):
    if subset not in _LS_CACHE:
        ds = (load_dataset("openslr/librispeech_asr", subset, split="test", streaming=True)
              .cast_column("audio", Audio(sampling_rate=16000)))
        smp = list(ds.take(n))
        arrays = [np.asarray(s["audio"]["array"], dtype=np.float32) for s in smp]
        _LS_CACHE[subset] = (arrays, [s["text"] for s in smp], [s["id"] for s in smp],
                             sum(len(a) for a in arrays) / TARGET_SR)
    return _LS_CACHE[subset]

def run_librispeech(tag, model_name, pipe):
    for subset, n in {"clean": 2620, "other": 2939}.items():
        arrays, refs_, ids_, secs = get_librispeech(subset, n)
        t0 = time.time(); hyp = transcribe_many(pipe, arrays); el = time.time() - t0
        fn = f"{RESULTS_DIR}/librispeech_test_{subset}__{tag}.json"
        save_results(fn, model=tag, dataset=f"librispeech test-{subset} (full)", ids=ids_,
                     preds=hyp, refs=refs_, audio_seconds=secs, elapsed_seconds=el, lang="en")
        print_score(f"{tag} | test-{subset}", score(hyp, refs_, normalize_en))
        backup_file(fn)

def run_apptek(tag, pipe, rows=range(20)):
    ids_, refs_, hyp, secs = [], [], [], 0.0
    t0 = time.time()
    for r in rows:
        row = apptek_metadata[r]
        p = download_apptek_audio(row["file_name"], accent=row.get("accent"), cache_dir="/content/apptek_audio")
        if p is None: continue
        a = load_audio_file(p); secs += len(a) / TARGET_SR
        hyp.append(transcribe_ctc(pipe, a)); refs_.append(row["text"]); ids_.append(row["file_name"])
    fn = f"{RESULTS_DIR}/apptek_rows0-19__{tag}.json"
    save_results(fn, model=tag, dataset="apptek test rows 0-19", ids=ids_, preds=hyp, refs=refs_,
                 audio_seconds=secs, elapsed_seconds=time.time() - t0, lang="en")
    print_score(f"{tag} | apptek", score(hyp, refs_, normalize_en))
    backup_file(fn)

def free(pipe):
    del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
LARGE = "facebook/wav2vec2-large-960h-lv60-self"
pipe = load_ctc_pipeline(LARGE, batch_size=8)        # large models support batching
run_librispeech("large_lv60_greedy", LARGE, pipe)
run_apptek("large_lv60_greedy", pipe)
free(pipe)

**C.3 (Optional, not used for the final numbers) n-gram language model decoding.** Beam search with a KenLM 3-gram LM via `pyctcdecode`. The `kenlm` wheel failed to build on Colab, so the final results use greedy decoding. Set `RUN_LM = True` only if `import kenlm` works in your runtime.

In [ ]:
RUN_LM = False   # kenlm did not build on Colab in our runs; greedy decoding is used for all reported numbers
if RUN_LM:
    import subprocess
    subprocess.run("pip install -q pyctcdecode kenlm", shell=True, check=True)
    subprocess.run("mkdir -p /content/lm && wget -q -nc https://www.openslr.org/resources/11/3-gram.pruned.1e-7.arpa.gz -P /content/lm "
                   "&& gunzip -kf /content/lm/3-gram.pruned.1e-7.arpa.gz", shell=True, check=True)
    LM = "/content/lm/3-gram.pruned.1e-7.arpa"
    pipe = load_ctc_pipeline(LARGE, batch_size=8, lm_path=LM, alpha=0.5, beta=1.5)
    run_librispeech("large_lv60_3gram", LARGE, pipe)
    run_apptek("large_lv60_3gram", pipe)
    free(pipe)

**C.4 English results table**

In [ ]:
compare(sorted(glob.glob(RESULTS_DIR + "/*.json")))

## Part D. Hindi
**D.1 Test data and baselines.** Full FLEURS `hi_in` test set (418 clips). Whisper-large-v3 is the baseline.
Note: the zero-shot run of the original Vakyansh checkpoint is disabled by default. Its raw decoding gave a degenerate result (WER above 800%, almost all insertions), which is a decoding artefact and not a real score, so it is not reported.

In [ ]:
from datasets import load_dataset, Audio
HI_DS, HI_CFG = "google/fleurs", "hi_in"
HI_BASE = "Harveenchadha/vakyansh-wav2vec2-hindi-him-4200"

hi_test = load_dataset(HI_DS, HI_CFG, split="test").cast_column("audio", Audio(sampling_rate=16000))
hi_arrays = [np.asarray(r["audio"]["array"], dtype=np.float32) for r in hi_test]
hi_refs   = [r["transcription"] for r in hi_test]
hi_ids    = [r["id"] for r in hi_test]
hi_secs   = sum(len(a) for a in hi_arrays) / TARGET_SR
print("FLEURS hi_in test utterances:", len(hi_test), f"| {hi_secs/60:.1f} min audio")

def save_hi(tag, hyp, elapsed):
    fn = f"{RESULTS_DIR}/fleurs_hi_test__{tag}.json"
    save_results(fn, model=tag, dataset="fleurs hi_in test (full)", ids=hi_ids, preds=hyp, refs=hi_refs,
                 audio_seconds=hi_secs, elapsed_seconds=elapsed, lang="hi")
    print_score(f"{tag} | FLEURS hi test", score(hyp, hi_refs, normalize_hi))
    backup_file(fn)

# Disabled: degenerate output (WER > 800%, mostly insertions) = decoding artefact, not a real baseline.
RUN_ZERO_SHOT = False
if RUN_ZERO_SHOT:
    pipe = load_ctc_pipeline(HI_BASE, batch_size=8)
    t0 = time.time(); hyp = transcribe_many(pipe, hi_arrays); save_hi("vakyansh_zero_shot", hyp, time.time() - t0)
    del pipe; gc.collect(); torch.cuda.empty_cache()

**D.2 Whisper-large-v3 on the same 418 Hindi clips** (batch 2 with SDPA attention; a larger batch ran out of memory on a 15 GB T4).

In [ ]:
import gc
import time
import torch
from transformers import pipeline

# 1. Load with SDPA (memory-efficient attention) and low_cpu_mem_usage
wp = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    device=0,
    torch_dtype=torch.float16,
    model_kwargs={"attn_implementation": "sdpa", "low_cpu_mem_usage": True},
)

# 2. Use a generator expression (...) instead of a list [...] so audio isn't stacked all at once
audio_stream = ({"raw": a, "sampling_rate": TARGET_SR} for a in hi_arrays)

t0 = time.time()

# 3. Drop batch_size to 2 or 4, and process inside inference_mode()
texts = []
with torch.inference_mode():
    for out in wp(
        audio_stream,
        batch_size=2,
        chunk_length_s=30,
        generate_kwargs={"language": "hindi", "task": "transcribe"},
    ):
        texts.append(out["text"])

save_hi("whisper_large_v3_hi", texts, time.time() - t0)

# 4. Clean up
del wp
gc.collect()
torch.cuda.empty_cache()

**D.3 Hindi fine-tuning.** All 2,120 FLEURS training clips for 15 epochs (the first pilot saw under one epoch), missing Devanagari characters appended to the vocabulary while keeping the old output weights, SpecAugment, frozen feature encoder, effective batch 8 (4 x 2 accumulation), fp16. Best epoch chosen on the validation set; the test set is untouched until the end. Every epoch is pushed to a private Hub repo so a disconnect cannot lose progress. Takes roughly 40-60 minutes on a T4.

In [ ]:
import inspect, jiwer
from dataclasses import dataclass
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, TrainingArguments, Trainer

HI_EPOCHS, HI_LR, MAX_SEC = 15, 5e-5, 20
HI_OUT   = "/content/hindi_w2v_ft"
HI_FINAL = HI_OUT + "/final_model"
HUB_MODEL = f"{BACKUP_REPO.split('/')[0]}/hindi-w2v-ft"

hi_train = load_dataset(HI_DS, HI_CFG, split="train").cast_column("audio", Audio(sampling_rate=16000))
hi_valid = load_dataset(HI_DS, HI_CFG, split="validation").cast_column("audio", Audio(sampling_rate=16000))
print("train / validation clips:", len(hi_train), len(hi_valid))

processor = Wav2Vec2Processor.from_pretrained(HI_BASE)
tok = processor.tokenizer

# --- vocabulary: add characters the base model has never seen ---
chars = {c for t in hi_train["transcription"] for c in normalize_hi(t) if not c.isspace()}
missing = sorted(c for c in chars if c not in tok.get_vocab())
print("characters missing from base vocab:", len(missing), "".join(missing)[:60])
if missing:
    tok.add_tokens(missing)
assert max(tok.get_vocab().values()) + 1 == len(tok), "vocab ids are not contiguous"

model = Wav2Vec2ForCTC.from_pretrained(HI_BASE, ctc_loss_reduction="mean", pad_token_id=tok.pad_token_id,
                                       ctc_zero_infinity=True, mask_time_prob=0.05)
old_head, new_size = model.lm_head, len(tok)
if new_size != old_head.out_features:          # keep old rows, append freshly initialised rows
    new_head = torch.nn.Linear(old_head.in_features, new_size)
    with torch.no_grad():
        new_head.weight[:old_head.out_features] = old_head.weight
        new_head.bias[:old_head.out_features] = old_head.bias
        new_head.weight[old_head.out_features:].normal_(0.0, 0.02)
        new_head.bias[old_head.out_features:] = old_head.bias.min()
    model.lm_head, model.config.vocab_size = new_head, new_size
model.freeze_feature_encoder()
print("vocab size:", model.config.vocab_size)

# --- features / labels ---
def prep(b):
    b["input_values"] = processor.feature_extractor(b["audio"]["array"], sampling_rate=16000).input_values[0]
    b["input_length"] = len(b["input_values"])
    b["labels"] = tok(normalize_hi(b["transcription"])).input_ids
    return b

tr = hi_train.map(prep, remove_columns=hi_train.column_names)
va = hi_valid.map(prep, remove_columns=hi_valid.column_names)
n0 = len(tr); tr = tr.filter(lambda x: x["input_length"] < MAX_SEC * 16000)
print(f"dropped {n0 - len(tr)} train clips longer than {MAX_SEC}s; training on {len(tr)}")

@dataclass
class CTCCollator:
    processor: Wav2Vec2Processor
    def __call__(self, feats):
        batch = self.processor.pad([{"input_values": f["input_values"]} for f in feats],
                                   padding=True, return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats],
                                           padding=True, return_tensors="pt")
        batch["labels"] = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        return batch

def compute_metrics(p):
    ids = np.argmax(p.predictions, axis=-1)
    p.label_ids[p.label_ids == -100] = tok.pad_token_id
    hyp = [normalize_hi(x) for x in tok.batch_decode(ids)]
    ref = [normalize_hi(x) for x in tok.batch_decode(p.label_ids, group_tokens=False)]
    keep = [i for i, r in enumerate(ref) if r]
    R, H = [ref[i] for i in keep], [hyp[i] for i in keep]
    return {"wer": jiwer.wer(R, H), "cer": jiwer.cer(R, H)}

valid_params = inspect.signature(TrainingArguments.__init__).parameters
ev_key = "eval_strategy" if "eval_strategy" in valid_params else "evaluation_strategy"

raw_args = {
    "output_dir": HI_OUT,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": HI_EPOCHS,
    "learning_rate": HI_LR,
    "warmup_steps": 0.1,
    "weight_decay": 0.005,
    "fp16": True,
    "remove_unused_columns": False,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "wer",
    "greater_is_better": False,
    "logging_steps": 50,
    "report_to": "none",
    "push_to_hub": True,
    "hub_model_id": HUB_MODEL,
    "hub_strategy": "every_save",
    "hub_private_repo": True,
    ev_key: "epoch",
}

args = TrainingArguments(**{k: v for k, v in raw_args.items() if k in valid_params})

tk = {"processing_class": processor} if "processing_class" in inspect.signature(Trainer.__init__).parameters else {"tokenizer": processor}
trainer = Trainer(model=model, args=args, data_collator=CTCCollator(processor), compute_metrics=compute_metrics,
                  train_dataset=tr, eval_dataset=va, **tk)
trainer.train()

trainer.save_model(HI_FINAL); processor.save_pretrained(HI_FINAL)
backup_folder(HI_FINAL, "hindi_final")

**D.4 Evaluate the fine-tuned Hindi model with the same harness, then compare**

In [ ]:
del trainer, model; gc.collect(); torch.cuda.empty_cache()
pipe = load_ctc_pipeline(HI_FINAL, batch_size=8)
t0 = time.time(); hyp = transcribe_many(pipe, hi_arrays); save_hi("vakyansh_finetuned_v2", hyp, time.time() - t0)
del pipe; gc.collect(); torch.cuda.empty_cache()
compare(sorted(glob.glob(RESULTS_DIR + "/fleurs_hi*.json")))

## Part E. Freeze and export
Saves the English and Hindi models as clean self-contained folders, checks they reload and reproduce the saved predictions, writes a manifest (metrics + inference settings), and pushes everything to one private Hub repo that the FastAPI backend downloads from.

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from huggingface_hub import snapshot_download
import json, shutil

EXPORT = "/content/export"
EN_ID  = "facebook/wav2vec2-large-960h-lv60-self"
HI_LOCAL = "/content/hindi_w2v_ft/final_model"
if not os.path.exists(HI_LOCAL):          # runtime was reset -> fetch our fine-tune from the Hub backup
    snapshot_download(BACKUP_REPO, allow_patterns=["hindi_final/*"], local_dir="/content/hub")
    HI_LOCAL = "/content/hub/hindi_final"
if not glob.glob(RESULTS_DIR + "/*.json"):  # restore saved results too
    snapshot_download(BACKUP_REPO, allow_patterns=["*.json"], local_dir=RESULTS_DIR)

for name, src in {"en": EN_ID, "hi": HI_LOCAL}.items():
    p = Wav2Vec2Processor.from_pretrained(src); m = Wav2Vec2ForCTC.from_pretrained(src)
    m.save_pretrained(f"{EXPORT}/{name}", safe_serialization=True); p.save_pretrained(f"{EXPORT}/{name}")
    print(name, "->", f"{EXPORT}/{name}", "| vocab:", m.config.vocab_size, "| params:", f"{sum(x.numel() for x in m.parameters())/1e6:.0f}M")
shutil.copy("/content/asr_common.py", f"{EXPORT}/asr_common.py")

In [ ]:
import gc
import json
import numpy as np
import torch
from datasets import load_dataset, Audio

def free(*objs):
    for obj in objs:
        del obj
    gc.collect()
    gc.collect()
    torch.cuda.empty_cache()

# English: first 5 test-clean utterances
ds = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True).cast_column("audio", Audio(sampling_rate=16000))
en5 = list(ds.take(5))
pe = load_ctc_pipeline(f"{EXPORT}/en", batch_size=8)
en_hyp = transcribe_many(pe, [np.asarray(s["audio"]["array"], dtype=np.float32) for s in en5])
en_saved = json.load(open(f"{RESULTS_DIR}/librispeech_test_clean__large_lv60_greedy.json"))["preds_raw"][:5]
print("EN identical to saved run:", sum(normalize_en(a) == normalize_en(b) for a, b in zip(en_hyp, en_saved)), "/ 5")
print("  sample:", en_hyp[0][:90])
del pe; free()

# Hindi: first 5 FLEURS test clips
hi5 = load_dataset("google/fleurs", "hi_in", split="test").cast_column("audio", Audio(sampling_rate=16000)).select(range(5))
ph = load_ctc_pipeline(f"{EXPORT}/hi", batch_size=8)
hi_hyp = transcribe_many(ph, [np.asarray(r["audio"]["array"], dtype=np.float32) for r in hi5])
hi_saved = json.load(open(f"{RESULTS_DIR}/fleurs_hi_test__vakyansh_finetuned_v2.json"))["preds_raw"][:5]
print("HI identical to saved run:", sum(normalize_hi(a) == normalize_hi(b) for a, b in zip(hi_hyp, hi_saved)), "/ 5")
print("  sample:", hi_hyp[0][:90])
del ph; free()

In [ ]:
metrics = {}
for p in sorted(glob.glob(RESULTS_DIR + "/*.json")):
    s = rescore(p)
    if s.get("n"): metrics[f"{s['model']} | {s['dataset']}"] = {"n": s["n"], "wer": s["wer"], "cer": s["cer"]}

manifest = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "models": {
        "en": {"dir": "en", "base": EN_ID, "normalizer": "normalize_en", "fine_tuned": False},
        "hi": {"dir": "hi", "base": "Harveenchadha/vakyansh-wav2vec2-hindi-him-4200", "normalizer": "normalize_hi", "fine_tuned": True},
    },
    "inference": {"sampling_rate": 16000, "chunk_length_s": 20, "stride_length_s": [4, 2], "decoding": "greedy_ctc", "batch_size": 8},
    "metrics": metrics,
}
json.dump(manifest, open(f"{EXPORT}/manifest.json", "w"), indent=2, ensure_ascii=False)

FINAL_REPO = f"{BACKUP_REPO.split('/')[0]}/asr-final"
_api.create_repo(FINAL_REPO, repo_type="model", private=True, exist_ok=True)
_api.upload_folder(folder_path=EXPORT, repo_id=FINAL_REPO, repo_type="model")
print("FROZEN MODELS PUSHED ->", FINAL_REPO)
print(json.dumps(metrics, indent=1)[:1500])

## Part F. Remaining comparisons
These two cells score Whisper-large-v3 and NVIDIA NeMo on **English** with exactly the same files and normaliser as the wav2vec2 results above, and save to the same results folder, so `compare(...)` produces one comparable table. If something errors, the helper functions in `asr_common.py` are the reference for scoring.

### F.1 Whisper-large-v3 on English (LibriSpeech test-clean + AppTek rows 0-19)
Requires Part A, B and the English helper cell in C.2. Whisper's own 30 s chunked decoding is used for the long AppTek recordings. Check the S/D/I breakdown: very high deletions or insertions point to long-form decoding artefacts rather than real recognition quality.

In [ ]:
LS_LIMIT = None        # None = full test-clean (2,620 utterances, roughly 20-25 min on a T4); or e.g. 500 for a quick check
RUN_APPTEK = True      # 20 long recordings; slow (tens of minutes)
from transformers import pipeline

wp = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", device=0, torch_dtype=torch.float16,
              model_kwargs={"attn_implementation": "sdpa", "low_cpu_mem_usage": True})
gen = {"language": "english", "task": "transcribe"}

# LibriSpeech test-clean (arrays are cached by get_librispeech from cell C.2)
arrays, refs_, ids_, secs = get_librispeech("clean", 2620)
if LS_LIMIT:
    arrays, refs_, ids_ = arrays[:LS_LIMIT], refs_[:LS_LIMIT], ids_[:LS_LIMIT]
    secs = sum(len(a) for a in arrays) / TARGET_SR
t0, hyp = time.time(), []
with torch.inference_mode():
    for o in wp(({"raw": a, "sampling_rate": TARGET_SR} for a in arrays), batch_size=8, generate_kwargs=gen):
        hyp.append(o["text"])
fn = f"{RESULTS_DIR}/librispeech_test_clean__whisper_large_v3_en.json"
save_results(fn, model="whisper_large_v3_en", dataset=f"librispeech test-clean ({len(arrays)} utts)", ids=ids_,
             preds=hyp, refs=refs_, audio_seconds=secs, elapsed_seconds=time.time() - t0, lang="en")
print_score("whisper_large_v3 | test-clean", score(hyp, refs_, normalize_en)); backup_file(fn)

if RUN_APPTEK:
    ids_, refs_, hyp, secs, t0 = [], [], [], 0.0, time.time()
    for r in range(20):                                   # same rows 0-19 as the wav2vec2 runs
        row = apptek_metadata[r]
        p = download_apptek_audio(row["file_name"], accent=row.get("accent"), cache_dir="/content/apptek_audio")
        if p is None: continue
        a = load_audio_file(p); secs += len(a) / TARGET_SR
        with torch.inference_mode():
            hyp.append(wp({"raw": a, "sampling_rate": TARGET_SR}, chunk_length_s=30, batch_size=4, generate_kwargs=gen)["text"])
        refs_.append(row["text"]); ids_.append(row["file_name"])
    fn = f"{RESULTS_DIR}/apptek_rows0-19__whisper_large_v3_en.json"
    save_results(fn, model="whisper_large_v3_en", dataset="apptek test rows 0-19", ids=ids_, preds=hyp, refs=refs_,
                 audio_seconds=secs, elapsed_seconds=time.time() - t0, lang="en")
    print_score("whisper_large_v3 | apptek", score(hyp, refs_, normalize_en)); backup_file(fn)

del wp; gc.collect(); torch.cuda.empty_cache()

### F.2 NVIDIA NeMo Conformer-CTC on English (run in a FRESH runtime)
Installing NeMo changes pinned versions of several packages (`requests`, `pyarrow`, ...). Run this section in a **new Colab runtime**: execute Part A (A.1 to A.4) and Part B, then the next two cells. Do not run it in the same runtime as training. NeMo's English model is an English-only comparator and is not used for Hindi.

In [ ]:
# Optional comparison model dependency (NVIDIA NeMo Conformer).
# Run this cell only when you intend to execute the NeMo comparison cell.
!pip install -q "nemo_toolkit[asr]"

In [ ]:
import tempfile, soundfile as sf
from datasets import load_dataset, Audio
from nemo.collections.asr.models import EncDecCTCModelBPE

NEMO_NAME = "nvidia/stt_en_conformer_ctc_large"
LS_LIMIT, CHUNK_S = None, 10          # None = full test-clean; AppTek recordings are cut into 10 s pieces
nemo = EncDecCTCModelBPE.from_pretrained(model_name=NEMO_NAME).eval().to("cuda" if torch.cuda.is_available() else "cpu")

def nemo_transcribe(arrays, batch_size=16):
    tmp = tempfile.mkdtemp(); paths = []
    for i, a in enumerate(arrays):
        p = f"{tmp}/{i}.wav"; sf.write(p, a, TARGET_SR); paths.append(p)
    out = nemo.transcribe(paths, batch_size=batch_size)
    if isinstance(out, tuple): out = out[0]                       # older NeMo returns (best, all)
    return [o.text if hasattr(o, "text") else str(o) for o in out]  # newer NeMo returns Hypothesis objects

# --- LibriSpeech test-clean ---
ds = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True).cast_column("audio", Audio(sampling_rate=16000))
smp = list(ds.take(LS_LIMIT or 2620))
arrays = [np.asarray(s["audio"]["array"], dtype=np.float32) for s in smp]
refs_, ids_ = [s["text"] for s in smp], [s["id"] for s in smp]
secs = sum(len(a) for a in arrays) / TARGET_SR
t0 = time.time(); hyp = nemo_transcribe(arrays); el = time.time() - t0
fn = f"{RESULTS_DIR}/librispeech_test_clean__nemo_conformer_ctc_large.json"
save_results(fn, model="nemo_conformer_ctc_large", dataset=f"librispeech test-clean ({len(arrays)} utts)", ids=ids_,
             preds=hyp, refs=refs_, audio_seconds=secs, elapsed_seconds=el, lang="en")
print_score("NeMo | test-clean", score(hyp, refs_, normalize_en)); backup_file(fn)

# --- AppTek rows 0-19 (same rows as the wav2vec2 / Whisper runs) ---
ids_, refs_, hyp, secs, t0 = [], [], [], 0.0, time.time()
for r in range(20):
    row = apptek_metadata[r]
    p = download_apptek_audio(row["file_name"], accent=row.get("accent"), cache_dir="/content/apptek_audio")
    if p is None: continue
    a = load_audio_file(p); secs += len(a) / TARGET_SR
    step = CHUNK_S * TARGET_SR
    pieces = [a[i:i + step] for i in range(0, len(a), step) if len(a[i:i + step]) > 0.5 * TARGET_SR]
    hyp.append(" ".join(nemo_transcribe(pieces))); refs_.append(row["text"]); ids_.append(row["file_name"])
fn = f"{RESULTS_DIR}/apptek_rows0-19__nemo_conformer_ctc_large.json"
save_results(fn, model="nemo_conformer_ctc_large", dataset="apptek test rows 0-19", ids=ids_, preds=hyp, refs=refs_,
             audio_seconds=secs, elapsed_seconds=time.time() - t0, lang="en")
print_score("NeMo | apptek", score(hyp, refs_, normalize_en)); backup_file(fn)

### F.3 Final comparison table (all saved runs)

In [ ]:
compare(sorted(glob.glob(RESULTS_DIR + "/*.json")))

## Part G. How to read the results, and limitations
- **Comparability.** WER/CER are only comparable when the same files and the same normaliser are used for every model. All numbers in Parts C to F satisfy this. Numbers from the Appendix were produced earlier with a different normaliser and different sample sets and must not be compared with them.
- **English.** The pretrained base model reproduces its published WER under our harness. The large backbone roughly halves the error. No English fine-tuning is part of the final system: continuing to train a model on data it was already trained on gave no measurable gain (Appendix A.2).
- **Hindi.** Our fine-tune reaches 19.33% WER / 7.20% CER on the full FLEURS test set, versus 28.75% / 10.82% for Whisper-large-v3. FLEURS is read speech and our model trained on FLEURS, so this is an in-domain advantage; our Hindi normaliser also does not unify numerals or English words in Latin script, which disadvantages Whisper somewhat. Real microphone speech will score worse.
- **Known limitations.** No language model (the `kenlm` build failed), no voice-activity detection, English output is UPPERCASE without punctuation (the server applies cosmetic casing), code-switched Hindi-English speech is not handled, and accent robustness depends on the training data.
- **Future work.** n-gram/neural LM decoding, more Hindi data (Common Voice, IndicVoices), noise and speed augmentation, VAD (Silero) for long recordings, punctuation restoration, language identification.